# Stage 3 技術實作：基於歷史訂單之互補商品關聯規則挖掘 (FP-Growth)

## 實作目標
本技術單元旨在實作系統架構中的 **Offline 離線運算模組**。透過分析 91APP 歷史子單資料 (`OrderSlave`)，利用 **FP-Growth 演算法** 挖掘商品之間的關聯規則（Market Basket Analysis），進而產出 `relation_product`（最常被同時購買之互補商品表），作為 Stage 3 組合推薦引擎的候選集（Candidate Generation）。

## 驗證規範 (Notice)
為了確保模型評估的嚴謹度，避免時間序列上的資料洩漏（Data Leakage），本實作將**嚴格僅使用前 80% 時間區間之訂單數據**作為訓練集，保留後 20% 時間區間供後續階段進行交叉驗證（Validation）。

In [36]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import gc

print("--- Data Preprocessing Pipeline Activated ---")
SHOP_ID = "zXQPxhiL90nRa1XbvctRfA=="
CHUNK_SIZE = 500_000
chunks = []
for chunk in pd.read_csv(f'./Order_TS_filtered_{SHOP_ID}.csv', chunksize=CHUNK_SIZE,
                         on_bad_lines='skip', low_memory=False,
                         usecols=['TradesGroupCode', 'OuterProductSkuCode', 'SalePageId', 'OrderDateTime', 'StatusDef']):
    chunks.append(chunk[chunk['StatusDef'] == 'Finish'])

df_clean = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

df_clean['OrderDateTime'] = pd.to_datetime(df_clean['OrderDateTime'], format='mixed')
df_clean = df_clean.dropna(subset=['OuterProductSkuCode'])
print(f"完成交易且有商品代碼筆數: {len(df_clean):,}")

# 建立 OuterProductSkuCode -> SalePageId 對照表
sku_to_salepage = (df_clean.dropna(subset=['SalePageId'])
                   .drop_duplicates('OuterProductSkuCode')
                   .set_index('OuterProductSkuCode')['SalePageId']
                   .to_dict())
print(f"有 SalePageId 對應的 SKU 數: {len(sku_to_salepage):,}")

df_clean = df_clean.sort_values('OrderDateTime').reset_index(drop=True)

split_idx = int(len(df_clean) * 0.8)
split_date = df_clean['OrderDateTime'].iloc[split_idx]
print(f"【時間分割點】: {split_date}")

df_train = df_clean.iloc[:split_idx].copy()
print(f"訓練集數據筆數 (前 80%): {len(df_train):,}")

del df_clean
gc.collect()

--- Data Preprocessing Pipeline Activated ---
完成交易且有商品代碼筆數: 4,016,035
有 SalePageId 對應的 SKU 數: 685
【時間分割點】: 2023-10-23 12:59:04.377000
訓練集數據筆數 (前 80%): 3,212,828


0

## 數據矩陣轉換 (One-Hot Encoding)
由於 FP-Growth 演算法要求輸入格式為佈林矩陣（Transaction Matrix），我們需要將同一筆主單編號 (`TradesGroupCode`) 下的所有子單商品 (`SalePageId`) 打包成購物籃，並進行 One-Hot 編碼轉換。

In [37]:
from mlxtend.preprocessing import TransactionEncoder
from collections import Counter

print("篩選多商品訂單...")
basket_list = df_train.groupby('TradesGroupCode')['OuterProductSkuCode'].apply(list)
basket_list = basket_list[basket_list.apply(len) >= 2]
print(f"含 2 件以上商品的訂單數: {len(basket_list):,}")

# 只保留 Top 500 最高頻商品，避免稀疏矩陣過大
product_counts = Counter(p for b in basket_list for p in b)
top_products = set(p for p, _ in product_counts.most_common(500))
basket_list = basket_list.apply(lambda b: [p for p in b if p in top_products])
basket_list = basket_list[basket_list.apply(len) >= 2]
print(f"限縮至 Top 500 商品後，有效訂單數: {len(basket_list):,}")

te = TransactionEncoder()
te_array = te.fit(basket_list).transform(basket_list, sparse=True)
basket_sets = pd.DataFrame.sparse.from_spmatrix(te_array, columns=te.columns_)
print(f"稀疏矩陣建立完成。訂單數: {basket_sets.shape[0]:,}, 商品數: {basket_sets.shape[1]:,}")

gc.collect()

篩選多商品訂單...
含 2 件以上商品的訂單數: 718,041
限縮至 Top 500 商品後，有效訂單數: 716,975
稀疏矩陣建立完成。訂單數: 716,975, 商品數: 500


0

## FP-Growth 演算法執行與關聯商品表 (`relation_product`) 產出
在此步驟中，我們設定最小支持度（`min_support`）與最小信賴度（`min_threshold`），透過 `mlxtend` 跑出商品間的強關聯規則，並將其格式化為系統架構所需的互補商品關聯對照表。

In [39]:
frequent_itemsets = fpgrowth(basket_sets, min_support=0.001, use_colnames=True)
print(f"頻繁模式樹建立成功，共挖掘出 {len(frequent_itemsets)} 組頻繁商品集。")

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0,
                          num_itemsets=len(frequent_itemsets))
print(f"關聯規則篩選完成，共生成 {len(rules)} 條有效規則。")

rules = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].copy()

rules['antecedent_id'] = rules['antecedents'].apply(lambda x: list(x)[0])
rules['consequent_id'] = rules['consequents'].apply(lambda x: list(x)[0])

relation_product = (rules[['antecedent_id', 'consequent_id', 'support', 'confidence', 'lift']]
                    .reset_index(drop=True))
relation_product.columns = ['target_product_id', 'complementary_product_id', 'support', 'confidence', 'lift']

relation_product['target_salepage_id'] = (relation_product['target_product_id']
                                          .map(sku_to_salepage)
                                          .astype('Int64'))
relation_product['complementary_salepage_id'] = (relation_product['complementary_product_id']
                                                  .map(sku_to_salepage)
                                                  .astype('Int64'))

# 過濾掉同一個 SalePageId 的規則
before = len(relation_product)
relation_product = relation_product[
    relation_product['target_salepage_id'].isna() |
    relation_product['complementary_salepage_id'].isna() |
    (relation_product['target_salepage_id'] != relation_product['complementary_salepage_id'])
].reset_index(drop=True)
print(f"過濾同 SalePageId 規則：{before} → {len(relation_product)} 筆")

# 讀取 SalePage 商品名稱
df_salepage = pd.read_csv('./SalePage.csv', usecols=['SalePageId', 'SalePageTitle'], encoding='utf-8-sig')
salepage_title = df_salepage.set_index('SalePageId')['SalePageTitle'].to_dict()

relation_product['target_title'] = relation_product['target_salepage_id'].map(salepage_title)
relation_product['complementary_title'] = relation_product['complementary_salepage_id'].map(salepage_title)

# 過濾掉滿額贈商品
before = len(relation_product)
relation_product = relation_product[
    ~relation_product['target_title'].fillna('').str.startswith('單筆滿') &
    ~relation_product['complementary_title'].fillna('').str.startswith('單筆滿')
].reset_index(drop=True)
print(f"過濾滿額贈規則：{before} → {len(relation_product)} 筆")

# 綜合分數：兼顧流行度(confidence)與真正互補性(lift)
relation_product['score'] = relation_product['confidence'] * np.log(relation_product['lift'])
relation_product = relation_product.sort_values('score', ascending=False).reset_index(drop=True)

# 欄位順序
cols = ['target_title', 'complementary_title',
        'target_product_id', 'complementary_product_id',
        'target_salepage_id', 'complementary_salepage_id',
        'score', 'confidence', 'lift', 'support']
relation_product = relation_product[cols]

relation_product.to_csv(f'./relation_product_{SHOP_ID}.csv', index=False)
print(f"\n--- Success: relation_product table generated successfully! ---")
print(f"共計 {len(relation_product):,} 筆強關聯規則。")
print(f"其中有 target_salepage_id: {relation_product['target_salepage_id'].notna().sum():,} 筆")
print(f"其中有 complementary_salepage_id: {relation_product['complementary_salepage_id'].notna().sum():,} 筆")

relation_product.head(20)

頻繁模式樹建立成功，共挖掘出 915 組頻繁商品集。
關聯規則篩選完成，共生成 970 條有效規則。
過濾同 SalePageId 規則：832 → 764 筆
過濾滿額贈規則：764 → 606 筆

--- Success: relation_product table generated successfully! ---
共計 606 筆強關聯規則。
其中有 target_salepage_id: 606 筆
其中有 complementary_salepage_id: 606 筆


,target_title,complementary_title,target_product_id,complementary_product_id,target_salepage_id,complementary_salepage_id,score,confidence,lift,support
0,媽媽好美~多送您一盒玉容面膜,全新玉容亮白面膜-升級款（兩盒特價）共10片,TIS7SECvJ5dDBEHRWbIGHw==,pTHi4v1uctatozC/Ioqijw==,7762691,6436896,5.485133,1.000000,241.081036,0.001081
1,黑不溜丟弱酸洗髮皂 100g（單顆）,單顆送Q彈的矽膠皂架❤️,O3QQrP12+DpxfVh5nZTkqw==,R70epjHuOaGrK4fZN5n75g==,7754244,7762598,4.160030,0.871504,118.319768,0.001608
2,唐朝楊貴妃凝脂秘方 紅玉面膜*10片（紅綠面膜任選多件更便宜）,吹彈水潤益母草水感面膜*10片（紅綠面膜任選多件更便宜）,mwAAX/e3hLaGVCsQQTO1fA==,fFJjaFsQWzai3Us0yxDFOw==,4864686,5541728,3.205310,0.649980,138.572623,0.002318
3,益生元黑馬卡賦黑洗髮精450g 多瓶更優惠,益生元咖啡因養髮洗髮精450g 多瓶更優惠,bpbkSdMCBQgbz6SRkvhEiQ==,ELp0Vx8do+Fcjnwbn5zkqg==,8789625,8789890,2.960420,0.591693,148.904506,0.002106
4,唐朝楊貴妃凝脂秘方 紅玉面膜*10片（紅綠面膜任選多件更便宜）,吹彈水潤益母草水感面膜*10片（紅綠面膜任選多件更便宜）,0aqwT0rkDTMo6jfReu9GUA==,39YQYiEgfsQMGlfgxmAHng==,4864686,5541728,2.726043,0.645860,68.087809,0.004286
5,益生元咖啡因養髮洗髮精450g 多瓶更優惠,益生元黑馬卡賦黑洗髮精450g 多瓶更優惠,ELp0Vx8do+Fcjnwbn5zkqg==,bpbkSdMCBQgbz6SRkvhEiQ==,8789890,8789625,2.651804,0.530011,148.904506,0.002106
6,舒眠安神輕奢~薰衣草精油香氛天然蠟燭200ml/6.76 oz（任選三件85折）,活力元氣~甜橙精油天然蠟燭200ml/6.76 oz（任選三件85折）,gd1wA99UcuVHxUJ3KsgLjQ==,BkM88sC+9F78b3lHS77yCA==,6845319,6845313,2.640035,0.620963,70.212054,0.004049
7,吹彈水潤益母草水感面膜*10片（紅綠面膜任選多件更便宜）,唐朝楊貴妃凝脂秘方 紅玉面膜*10片（紅綠面膜任選多件更便宜）,fFJjaFsQWzai3Us0yxDFOw==,mwAAX/e3hLaGVCsQQTO1fA==,5541728,4864686,2.437103,0.494202,138.572623,0.002318
8,髮/香_香水檸檬頭皮淨化去屑洗髮露750g 多瓶更優惠,髮/香_小蒼蘭強韌髮根蓬鬆洗髮露750g 多瓶更優惠,VaOnKtUNFugCeHSiHTB/pQ==,2RZNDm4OJMe3mwffQ5OZ8Q==,8948096,8947983,2.231789,0.553819,56.250747,0.004299
9,*無患子手工手珠（三種尺寸）⭐董事長親自上陣【加購】,無患子手工手珠（三種尺寸）⭐董事長親自上陣,G/jpwznM+Jc0L2nNBPfzBg==,U9VOHIINN5IXSMyef8i3ag==,8922654,8922640,1.978361,0.426822,103.037390,0.001070


In [40]:
same = relation_product[
    relation_product['target_salepage_id'].notna() &
    relation_product['complementary_salepage_id'].notna() &
    (relation_product['target_salepage_id'] == relation_product['complementary_salepage_id'])
]
print(f"兩個 SalePageId 相同的規則數: {len(same)} 筆")
same

兩個 SalePageId 相同的規則數: 0 筆


,target_title,complementary_title,target_product_id,complementary_product_id,target_salepage_id,complementary_salepage_id,score,confidence,lift,support


In [41]:
target_products = set(relation_product['target_product_id'].unique())
comp_products = set(relation_product['complementary_product_id'].unique())

print(f"target 側不同商品數:        {len(target_products):,}")
print(f"complementary 側不同商品數: {len(comp_products):,}")
print(f"兩側聯集（總不同商品數）:    {len(target_products | comp_products):,}")
print(f"兩側都出現的商品數:          {len(target_products & comp_products):,}")

target 側不同商品數:        145
complementary 側不同商品數: 145
兩側聯集（總不同商品數）:    145
兩側都出現的商品數:          145
